# 07 Wwpgd Diagnostics

In [ ]:
RESULTS_ROOT = ""
OUTPUT_ROOT = ""
ANALYSIS_PLAN = ""
PROFILE = ""
LEVEL = None
TOKEN_MULTIPLIER = None
BASE_OPTIMIZER = "adamw"
STRICT = False
RUN_ANALYSIS = False
REUSE_EXISTING_ANALYSIS = True
FIGURE_FORMAT = "png"

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from wwgpt.notebook_support import *
P = resolve_notebook_parameters(globals())
validate_paths(P)
print("Resolved notebook parameters:", json.dumps(P.summary(), indent=2))
print("Package provenance:", json.dumps(package_provenance(), indent=2))
runs = discover_completed_runs(P.results_root)
if P.level is not None and "level" in runs: runs = runs[runs.level.fillna(P.level).astype(int) == P.level]
if P.token_multiplier is not None and "token_multiplier" in runs: runs = runs[runs.token_multiplier.fillna(P.token_multiplier).astype(int) == P.token_multiplier]
pairs = pair_arms(runs, P.base_optimizer)
print(f"Discovered {len(runs)} completed runs and {len(pairs)} complete pairs")

In [ ]:
all_diag=[]; endpoint=[]; relax=[]; skips=[]
for _,r in runs[runs.arm.str.endswith('_wwpgd')].iterrows():
 run=Path(r.run_dir); d=load_wwpgd_internal_diagnostics(run)
 if d is not None: d=d.assign(seed=r.seed,arm=r.arm); all_diag.append(d)
 for name,target in [('wwpgd_endpoint_measurements.csv',endpoint),('wwpgd_endpoint_relaxation.csv',relax),('wwpgd_controller.csv',skips)]:
  x=load_wwpgd_artifact(run,name)
  if x is not None: x=x.assign(seed=r.seed,arm=r.arm); target.append(x)
diag=pd.concat(all_diag,ignore_index=True) if all_diag else pd.DataFrame(); ep=pd.concat(endpoint,ignore_index=True) if endpoint else pd.DataFrame(); dose=pd.concat(relax,ignore_index=True) if relax else pd.DataFrame(); ctl=pd.concat(skips,ignore_index=True) if skips else pd.DataFrame()
private=['k_pl','k_detx','k_star','selected_tail_size','selected_lambda_threshold','TraceLog_before','TraceLog_after','cayley_raw_ratio','cayley_applied_ratio']
compat=diag.get('diagnostics_mode',pd.Series(dtype=str)).eq('compatibility')
fabricated=bool(diag.loc[compat,[c for c in private if c in diag]].notna().any(axis=None)) if len(diag) and any(c in diag for c in private) else False
unsupported=diag.get('unsupported_internal_fields',pd.Series(dtype=str)); print('UNSUPPORTED INTERNAL FIELDS:', '; '.join(sorted(set(unsupported.dropna().astype(str)))))
health=pd.DataFrame([{'diagnostic_rows':len(diag),'compatibility_rows':int(compat.sum()),'native_rows':int(diag.get('diagnostics_mode',pd.Series(dtype=str)).eq('native').sum()),'fabricated_private_compatibility_values':fabricated,'warning':'compatibility diagnostics never reconstruct midpoint, Cayley, or TraceLog internals'}])
write_table(P,'wwpgd_diagnostic_health.csv',health)
write_table(P,'wwpgd_dose_by_layer.csv',dose.groupby('layer_name',dropna=False).agg(cumulative_applied_movement=('applied_relative_frobenius_change','sum')).reset_index() if len(dose) and 'layer_name' in dose and 'applied_relative_frobenius_change' in dose else pd.DataFrame(columns=['layer_name','cumulative_applied_movement']))
write_table(P,'wwpgd_endpoint_summary.csv',ep)
write_table(P,'wwpgd_skip_reason_summary.csv',ctl.get('skip_reason',pd.Series(dtype=str)).value_counts(dropna=False).rename_axis('skip_reason').reset_index(name='count'))
write_table(P,'wwpgd_diagnostic_capability.csv',pd.DataFrame([{'native_private_diagnostics_available':bool((diag.get('diagnostics_mode',pd.Series(dtype=str))=='native').any()),'compatibility_available':bool(compat.any()),'unsupported_internal_fields':';'.join(sorted(set(unsupported.dropna().astype(str))))}]))
display(health)
if P.strict and fabricated: raise RuntimeError('compatibility fields contain fabricated private values')